## Résultats réellement exécutés — 14 octobre 2026

Deux entraînements de **600 mises à jour** terminés sur **NVIDIA L4**, **20 860 416 paramètres** chacun. Parent étape 8 : **16/40**. Choix figé sur validation : **expanded_bible**.

| Bras | Meilleur checkpoint | Régression connue | chrF++ |
|---|---:|---:|---:|
| pilot_control | 100 | 21/40 | 61.084 |
| expanded_bible | 450 | 35/40 | 88.798 |

Les 40 questions sont une régression déjà observée sur des réponses connues. Ce résultat ne mesure pas la précision générale en conversation. Les résultats des tests bibliques et des cinq nouvelles réponses écrites sont conservés dans le rapport et ci-dessous. Aucun jugement de fluidité native.

**Tous les checkpoints restent dans Drive**, avec les meilleurs poids et les états complets de reprise séparés.

| Autres contrôles — réponses exactes attendues | Parent | Contrôle | Bible élargie |
|---|---:|---:|---:|
| Questions bibliques connues (8) | 0/8 | 0/8 | 0/8 |
| Passages bibliques réservés (12) | 0/12 | 0/12 | 0/12 |
| Reformulations quotidiennes réservées (16) | 0/16 | 0/16 | 0/16 |
| Nouveau dialogue écrit (5) | 0/5 | 0/5 | 0/5 |

Les 14 réponses simples d’entraînement restent exactes. **19 correspondances corrigées et aucune perdue sur les 40 questions connues**, mais les nouvelles réponses de dialogue restent décalées par rapport au contexte (notamment des mentions d’Aka ou de Konan dans les questions sur les plantes). Le score 35/40 ne constitue donc pas une validation de la conversation libre.


# MandiaCompact — étape 9 : correspondances et questions ancrées

**État : exécuté ; résultats réels ci-dessus.** Deux continuations du modèle Mandia de 20 860 416 paramètres, 600 mises à jour chacune sur L4. Aucun poids préentraîné externe.

Le français est rédigé par l’assistant à partir de textes alignés ; les réponses baoulé restent celles des sources. 843 exemples bibliques (631 TRAIN), dont 125 vues de questions sémantiques et 718 de reformulation contextualisée ; 786 vues du quotidien (554 TRAIN) ; 16 réponses issues de 4 nouveaux dialogues écrits SMOL (8 TRAIN, 3 validation, 5 test). Ces derniers sont des traductions publiées de dialogues écrits, pas des conversations enregistrées.

Le contrôle utilise le petit pilote biblique précédent. Le candidat utilise les vues bibliques étendues. Les autres données, poids de groupes, architecture, taux d’apprentissage et nombres de mises à jour sont identiques ; les tokens et le calcul diffèrent. Sélection sur validation avant génération des tests.

**16/40 est une mesure de régression déjà observée sur des réponses connues, pas une précision de conversation.** Les nouveaux tests sont minuscules et sans relecture native. La Bible entière n’a pas été importée ; l’édition du corpus mtBCI n’est pas identifiée et sa filiation avec Louis Segond n’est pas supposée.

**Stockage : tous les checkpoints complets, optimiseur et états aléatoires sont écrits dans ce dossier Drive.** Le notebook requiert les dossiers parents du projet, avec leurs SHA. Aucun gros fichier n’est téléchargé sur le Mac.


In [ ]:
# Training is private in Colab; persistent files and full checkpoints in Drive.
from google.colab import drive
import pathlib,sys,hashlib,subprocess,json,importlib
if not pathlib.Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
assert pathlib.Path('/content/drive/MyDrive').exists(), 'Connect the project Google Drive first'
ROOT9=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape9')
assert ROOT9.is_dir(), 'Project stage9 assets must exist in Drive'
ASSET_HASHES9={'quotidien-contextuel.jsonl': 'bace73aab9ab821ed4859ea45830b2d8d17634e359af309b2904a3c2c618ec76', 'AUDIT-DONNEES.json': '8546a73e88ebd0c8630fcc180f629339d0ed0cb168a4fc2d0ea3467484fd3a35', 'bible-etendue.jsonl': '36bf8130fa9c785e3e1f4ff931903f747c0bc1c064782e3146ba4200618b426c', 'mandia_stage9.py': 'dd516392c059b7f5b1c2609351bba3da93d87c611fb94ba9245ee8045a33c0d9', 'preparer-donnees.py': '6a2739ecccfe568cd08c30e617d791ea9cb6d1e362f536512d7b4337a90c4dfd', 'dialogues-ecrits-ajout.jsonl': '2e7a3f0634c96f0f8c91944db74133ca4ff0d0bc968baa4987f208a82782809a'}
FROZEN9={'mandia-compact-s2s-etape4/parallel-prepare.jsonl': 'dc2bfd9ba783c5edec159ec35bfbb66206598520bef31f960da3ad92cc240cbd', 'mandia-compact-s2s-etape4/replies-prepare.jsonl': '29334943cd507b8ffa9385b530b9625aec82c0741aeade3575a30c7df7e2caf1', 'mandia-compact-s2s-etape4/data/waxal-baoule-text.jsonl': '4f7730d32d3324d2cc819f33d7633b2d59e01f81c8e2f38651d0377050ec69d4', 'mandia-compact-s2s-etape5/quotidien-prepare.jsonl': '370d3bb631a4ad4765fbe3b15e0f6019c0470ce03c687f87ac71f6fda6b3048c', 'mandia-compact-s2s-etape6/dialogues-prepare.jsonl': '9375c6a0918a5bf8056cecd42df0c8aff90c3439220bc45bfe10e8cc700a2184', 'mandia-compact-s2s-etape7/teacher-grounded-qa.jsonl': '3cfe3c27ed8e221cad480755f9ee1a0fcee1c67583f58744cda427932e94910b', 'mandia-compact-s2s-etape7/mandia_stage7.py': '6044fb5633f0d0438000f005849f78db20c3b6c5edc2b693530818b53322ea41', 'mandia-compact-s2s-etape8/bible-qa-pilote.jsonl': '868ea39a2961c3b590260b3a7fab51581b82aed7db75726c4ab7369d1ede6a0e', 'mandia-compact-s2s-etape8/wide_20m/model.pt': '6f2bac99307900f96d80a6b9d4980a3b5bc19fc07486beff456979b2d3decbee'}

for rel,expected in ASSET_HASHES9.items():
 assert hashlib.sha256((ROOT9/rel).read_bytes()).hexdigest()==expected, 'Stage9 source changed: '+rel
for rel,expected in FROZEN9.items():
 assert hashlib.sha256((ROOT9.parent/rel).read_bytes()).hexdigest()==expected, 'Frozen project asset changed: '+rel
try:import tokenizers,sacrebleu
except ImportError:subprocess.check_call([sys.executable,'-m','pip','install','-q','tokenizers==0.22.1','sacrebleu==2.5.1'])
for rel in ['mandia-compact-s2s-etape6/code','mandia-compact-s2s-etape7','mandia-compact-s2s-etape9']:sys.path.insert(0,str(ROOT9.parent/rel))
import mandia_stage9
importlib.reload(mandia_stage9)
print('MANDIA_ETAPE9_DEBUT: Drive et SHA verifies; 2 bras de 600 mises a jour, 20.86M chacun.',flush=True)
report9=mandia_stage9.main()


In [ ]:
#@title Essayer le checkpoint sélectionné — réponse expérimentale
message_fr = "Quel animal Konan possède-t-il ?" #@param {type:"string"}
contexte_fr = "Konan a une chèvre." #@param {type:"string"}
from google.colab import drive
import pathlib,sys,json,hashlib
if not pathlib.Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
runtime=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime')
sys.path.insert(0,str(runtime/'mandia-compact-s2s-etape6/code'))
import torch
from tokenizers import Tokenizer
from mandia_seq2seq import MandiaS2S,S2SConfig,generate
root=runtime/'mandia-compact-s2s-etape9'
summary=json.loads((root/'rapport-etape9.json').read_text());chosen=summary['selected_by_validation']
path=root/chosen/'model.pt'
receipt=json.loads((root/'SAUVEGARDE-VERIFIEE.json').read_text())
expected=next(r['sha256'] for r in receipt['files'] if r['path']==chosen+'/model.pt')
assert hashlib.sha256(path.read_bytes()).hexdigest()==expected
payload=torch.load(path,map_location='cpu',weights_only=True)
tp=runtime/'mandia-compact-s2s-etape4/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest()==payload['tokenizer_sha256']
tokenizer=Tokenizer.from_file(str(tp));dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=MandiaS2S(S2SConfig(**payload['config'])).to(dev);model.load_state_dict(payload['state_dict']);model.eval()
prompt=f"Contexte : {contexte_fr}\nMessage : {message_fr}" if contexte_fr.strip() else message_fr
source=[4]+tokenizer.encode(prompt).ids+[2]
assert len(source)<=160,'Raccourcir le contexte ; aucune troncature silencieuse'
with torch.inference_mode(): output,_=generate(model,source,1,2,96)
print('Checkpoint:',chosen,'; Baoulé expérimental :',tokenizer.decode(output))
print('Réponse à vérifier : erreurs de sens encore possibles ; exemple connu si valeurs par défaut.')
del model,payload
